# <span style="color:#e0bda8">1. Import Packages and Libraries</span>

In [55]:
# 1. Data Manipulation
# =====================================================
import pandas as pd
import numpy as np


# 2. Data Preprocessing
# =====================================================
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler, PowerTransformer
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline


# 3. Statistical Analysis
# =====================================================
from scipy.stats import skew, kurtosis


# 4. Path Configuration
# =====================================================
import os


# 5. Utilities
# =====================================================
import warnings

warnings.filterwarnings('ignore')

# <span style="color:#e0bda8">2. Project Structure and Directory Configuration </span>

In [56]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA = os.path.join(PROJECT_ROOT, "01_Data")
DATA_RAW = os.path.join(DATA, "01_Raw")
DATA_PROCESSED = os.path.join(DATA, "02_Processed")
DATA_STATS = os.path.join(DATA, "03_Statistics")
DATA_INDEX = os.path.join(DATA, "04_Index_Final")
FINAL_DATA = os.path.join(DATA, "05_Final_Data")

EDA = os.path.join(PROJECT_ROOT, "03_EDA")

CLUSTER_PCA = os.path.join(PROJECT_ROOT, "04_Clustering_PCA")
CLUSTER_PCA_EXCEL = os.path.join(CLUSTER_PCA, "01_Excel")
CLUSTER_PCA_FIG = os.path.join(CLUSTER_PCA, "02_Figures")

METHOD = os.path.join(PROJECT_ROOT, "05_Method_Comparison")
METHOD_EXCEL = os.path.join(METHOD, "01_Excel")
METHOD_FIG = os.path.join(METHOD, "02_Figures")

ML_RESULTS = os.path.join(PROJECT_ROOT, "06_ML_Results")
ML_METRICS = os.path.join(ML_RESULTS, "01_Metrics")
ML_PRED = os.path.join(ML_RESULTS, "02_Predictions")
ML_MODELS = os.path.join(ML_RESULTS, "03_Models")
ML_SHAP = os.path.join(ML_RESULTS, "04_SHAP")
ML_SHAP_EXCEL = os.path.join(ML_SHAP, "01_Excel")
ML_SHAP_FIG = os.path.join(ML_SHAP, "02_Figures")

FINAL_SCORES = os.path.join(PROJECT_ROOT, "07_Final_ESG_Scores")
FINAL_SCORES_EXCEL = os.path.join(FINAL_SCORES, "01_Excel")
FINAL_SCORES_FIG = os.path.join(FINAL_SCORES, "02_Figures")

# <span style="color:#e0bda8">3. Reading the Data </span>

In [57]:
df_env_filtered =  pd.read_csv(os.path.join(DATA_PROCESSED, "df_env_filtered.csv"))
df_env_filtered = df_env_filtered.set_index(['Economy', 'Year'])

df_soc_filtered =  pd.read_csv(os.path.join(DATA_PROCESSED, "df_soc_filtered.csv"))
df_soc_filtered = df_soc_filtered.set_index(['Economy', 'Year'])

df_gov_filtered =  pd.read_csv(os.path.join(DATA_PROCESSED, "df_gov_filtered.csv"))
df_gov_filtered = df_gov_filtered.set_index(['Economy', 'Year'])

In [58]:
df_env_filtered

PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)  \
Economy                Year                                                                           
Albania                2012                                            20.9627                        
                       2013                                            19.4075                        
                       2014                                            19.3679                        
                       2015                                            19.1216                        
                       2016                                            17.6022                        
                       2017                                            18.5552                        
                       2018                                            19.0462                        
                       2019                                            15.7047                        
                       2020                                            15.7070                        
Armenia                2012                                            43.2896                        
                       2013                                            41.0505                        
                       2014                                            38.6392                        
                       2015                                            38.1149                        
                       2016                                            35.8609                        
                       2017                                            34.0154                        
                       2018                                            34.2014                        
                       2019                                            32.6618                        
                       2020                                            30.5796                        
Austria                2012                                            14.7199                        
                       2013                                            14.6599                        
                       2014                                            12.9541                        
                       2015                                            13.6000                        
                       2016                                            12.5144                        
                       2017                                            12.1558                        
                       2018                                            12.5862                        
                       2019                                            11.4646                        
                       2020                                            10.9335                        
Belgium                2012                                            15.2150                        
                       2013                                            14.3698                        
                       2014                                            12.9004                        
                       2015                                            13.3390                        
                       2016                                            12.3939                        
                       2017                                            12.7692                        
                       2018                                            12.4926                        
                       2019                                            11.2144                        
                       2020                                            11.2161                        
Bosnia and Herzegovina 2012                                            35.5047                        
            

# <span style="color:#e0bda8">4. Pipeline Architecture </span>

## <span style="color:#e0bda8">4.1. Signal Aligning and Log-Transformation Decisions </span>

In [59]:
# 1. Negative variables (the lower the original value, the better the final score) - 24 variables

negative_vars = [
    'PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)',
    'Carbon dioxide (CO2) net fluxes from LULUCF - Total excluding non-tropical fires (Mt CO2e)',
    'Total greenhouse gas emissions per capita excluding LULUCF (t CO2e/capita)',
    'Electricity production from coal sources (% of total)',
    'Energy imports, net (% of energy use)',
    'Energy intensity level of primary energy (MJ/$2021 PPP GDP)',
    'Energy use (kg of oil equivalent per capita)',
    'Fossil fuel energy consumption (% of total)',
    'Cooling Degree Days',
    'Heating Degree Days',
    'Level of water stress: freshwater withdrawal as a proportion of available freshwater resources',
    'Adjusted savings, natural resources depletion (% of GNI)',
    'Adjusted savings, net forest depletion (% of GNI)',
    'Annual freshwater withdrawals, total (% of internal resources)',
    'Tree Cover Loss',
    'Unemployment, total (% of total labor force) (modeled ILO estimate)',
    'Mortality rate, under-5 (per 1,000 live births)',
    'Prevalence of overweight (% of adults)',
    'Prevalence of undernourishment (% of population)',
    'Gini index',
    'Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)',
    'Poverty headcount ratio at $8.30 a day (2021 PPP) (% of population)',
    'Poverty headcount ratio at national poverty lines (% of population)',
    'Share of youth not in education, employment or training, total (% of youth population)  (modeled ILO estimate)'
]

# 2. Centered variables (distance to the reference value) - 2 variables

centered_vars = [
    'Standardised Precipitation-Evapotranspiration Index', 
    'Fertility rate, total (births per woman)',  
]

ref_values = {
    'Standardised Precipitation-Evapotranspiration Index': 0,
    'Fertility rate, total (births per woman)': 2.1,
}

# 3. Positive variables (the higher the original value, the better the final score) - 26 variables

positive_vars = [
    'Renewable electricity output (% of total electricity output)',
    'Renewable energy consumption (% of total final energy consumption)',
    'Forest area (% of land area)',
    'Terrestrial and marine protected areas (% of total territorial area)',
    'Food production index (2014-2016 = 100)',
    'Access to clean fuels and technologies for cooking  (% of population)',
    'People using safely managed drinking water services (% of population)',
    'People using safely managed sanitation services (% of population)',
    'Life expectancy at birth, total (years)',
    'Government expenditure on education, total (% of government expenditure)',
    'School enrollment, primary (% gross)',
    'Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)',
    'Hospital beds (per 1,000 people)',
    'Individuals using the Internet (% of population)',
    'Proportion of seats held by women in national parliaments (%)',
    'Ratio of female to male labor force participation rate (%) (modeled ILO estimate)',
    'Government Effectiveness: Estimate',
    'Economic and Social Rights Performance Score',
    'Voice and Accountability: Estimate',
    'Patent applications, residents',
    'Research and development expenditure (% of GDP)',
    'Net migration',
    'Political Stability and Absence of Violence/Terrorism: Estimate',
    'Rule of Law: Estimate',
    'Agriculture, forestry, and fishing, value added (% of GDP)',
    'Wage and salaried workers, total (% of total employment) (modeled ILO estimate)'
]

# 4. Signed Log1p Transformation variables - 15 variables

transformation_vars = [
    'Carbon dioxide (CO2) net fluxes from LULUCF - Total excluding non-tropical fires (Mt CO2e)',
    'Tree Cover Loss', 
    'Energy imports, net (% of energy use)',
    'Energy intensity level of primary energy (MJ/$2021 PPP GDP)',
    'Energy use (kg of oil equivalent per capita)',
    'Adjusted savings, natural resources depletion (% of GNI)',
    'Adjusted savings, net forest depletion (% of GNI)',
    'Annual freshwater withdrawals, total (% of internal resources)',
    'Agriculture, forestry, and fishing, value added (% of GDP)', 
    'Patent applications, residents',  
    'Prevalence of undernourishment (% of population)',
    'Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)',
    'Poverty headcount ratio at $8.30 a day (2021 PPP) (% of population)', 
    'Mortality rate, under-5 (per 1,000 live births)' 
]

## <span style="color:#e0bda8">4.2. Classes Creation </span>

In [60]:
class TemporalInterpolationTransformer(BaseEstimator, TransformerMixin):
    """
    Custom transformer that performs time-based interpolation within groups.

    Missing values are filled using:
        1. Time interpolation (based on chronological order)
        2. Forward fill
        3. Backward fill

    Designed for panel data (e.g., country-year structure).
    """

    def __init__(self, group_col='Economy', time_col='Year', numeric_cols=None):
        self.group_col = group_col
        self.time_col = time_col
        self.numeric_cols = numeric_cols

    def fit(self, X, y=None):
        # Automatically detect numeric columns if not provided
        if self.numeric_cols is None:
            self.numeric_cols_ = X.select_dtypes(include='number').columns.tolist()
        else:
            self.numeric_cols_ = self.numeric_cols
        return self

    def transform(self, X):
        # Work on a copy and reset index for grouping
        X = X.copy().reset_index()

        # Ensure time column is datetime (required for interpolation)
        X['Year'] = pd.to_datetime(X['Year'], format='%Y')

        def interpolate_group(group):
            # Preserve group identifier
            group[self.group_col] = group.name

            original_index = group.index

            # Set time index for interpolation
            group = group.set_index(self.time_col)

            # Apply interpolation + forward/backward fill
            group[self.numeric_cols_] = (
                group[self.numeric_cols_]
                .interpolate(method='time')
                .ffill()
                .bfill()
            )

            # Restore original structure
            group = group.reset_index()
            group.index = original_index
            return group

        # Apply interpolation per group (e.g., per country)
        X = X.groupby(self.group_col, group_keys=False).apply(
            interpolate_group, include_groups=False
        )

        # Restore original index format
        X['Year'] = X['Year'].dt.year.astype(int)
        X = X.set_index(['Economy', 'Year'])

        return X

In [61]:
class KNNImputerSafe(BaseEstimator, TransformerMixin):
    """
    Wrapper around KNNImputer that preserves original non-missing values.

    Only missing values are replaced, ensuring no unintended changes
    to observed data (important for reproducibility and interpretability).
    """

    def __init__(self, n_neighbors=5, weights='distance'):
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.imputer = KNNImputer(
            n_neighbors=self.n_neighbors,
            weights=self.weights
        )

    def fit(self, X, y=None):
        # Fit imputer on full dataset
        self.imputer.fit(X)
        return self

    def transform(self, X):
        X_copy = X.copy()

        # Track missing values
        mask_missing = X_copy.isna()

        # Perform KNN imputation
        X_imputed_array = self.imputer.transform(X_copy)

        # Convert back to DataFrame
        X_imputed = pd.DataFrame(
            X_imputed_array,
            index=X_copy.index,
            columns=X_copy.columns
        )

        # Replace only missing entries (preserve original observed values)
        X_copy[mask_missing] = X_imputed[mask_missing]

        return X_copy

In [62]:
class SafePowerTransformer(BaseEstimator, TransformerMixin):
    """
    Applies a Power Transformation (e.g., Yeo-Johnson) safely to selected columns.

    - Skips columns with insufficient variation (constant or all missing)
    - Avoids transformation errors (e.g., optimization failures)
    - Preserves original values where transformation is not applicable
    """

    def __init__(self, columns=None, method='yeo-johnson'):
        self.columns = columns
        self.method = method
        self.transformers_ = {}

    def fit(self, X, y=None):
        # Initialize storage for fitted transformers
        self.transformers_ = {}

        # Determine which columns to process
        cols_to_process = self.columns if self.columns is not None else X.columns
        
        for col in cols_to_process:
            # Use only non-missing values for fitting
            series = X[col].dropna()
            
            # Check if there is enough variation to apply transformation
            if len(series) > 0 and series.nunique() > 1:
                try:
                    pt = PowerTransformer(method=self.method, standardize=True)

                    # Reshape required by sklearn (2D input)
                    pt.fit(series.values.reshape(-1, 1))

                    self.transformers_[col] = pt

                except Exception as e:
                    # Skip column if transformation fails
                    warnings.warn(
                        f"Column '{col}' failed the transformation ({type(e).__name__}). Keeping original values."
                    )
            else:
                # Skip columns with no variation
                warnings.warn(
                    f"Column '{col}' has constant or missing values. Skipping transformation."
                )
        
        return self

    def transform(self, X):
        X_copy = X.copy()

        for col, pt in self.transformers_.items():
            # Identify non-missing values
            mask = X_copy[col].notnull()

            if mask.any():
                # Apply transformation only to valid values
                vals = X_copy.loc[mask, col].values.reshape(-1, 1)
                X_copy.loc[mask, col] = pt.transform(vals).flatten()

        return X_copy

In [63]:
class SafeMinMaxScaler(BaseEstimator, TransformerMixin):
    """
    Applies Min-Max scaling to selected columns while preserving the rest of the dataset.

    Scales features to a specified range (default: 0–100).
    """

    def __init__(self, columns=None, feature_range=(0, 100)):
        self.columns = columns
        self.scaler = MinMaxScaler(feature_range=feature_range)
    
    def fit(self, X, y=None):
        # Fit scaler only on selected columns
        self.scaler.fit(X[self.columns])
        return self
    
    def transform(self, X):
        X_copy = X.copy()

        # Apply scaling only to specified columns
        X_copy[self.columns] = self.scaler.transform(X_copy[self.columns])

        return X_copy

In [64]:
class SignedLog1pTransformer(BaseEstimator, TransformerMixin):
    """
    Applies Signed Log1p transformation to selected variables.

    Transformation:
        signed_log1p(x) = sign(x) * log(1 + |x|)

    This preserves the sign of the original values while reducing skewness
    and compressing extreme values.
    """

    def __init__(self, vars_to_transform=None):
        self.vars_to_transform = vars_to_transform
    
    def fit(self, X, y=None):
        # No learning required for this transformation
        return self
    
    def transform(self, X):
        X_copy = X.copy()

        # Apply transformation only to selected variables
        for var in self.vars_to_transform or []:
            if var not in X_copy.columns:
                continue

            # Signed log1p transformation (handles negative values safely)
            X_copy[var] = np.sign(X_copy[var]) * np.log1p(np.abs(X_copy[var]))
        
        return X_copy

In [65]:
class SignalAligningTransformer(BaseEstimator, TransformerMixin):
    """
    Aligns variable signals to ensure consistent interpretability across features.

    - Inverts predefined negatively-oriented variables
    - Centers selected variables around reference values
    - Ensures consistent directionality for downstream modeling
    """

    def __init__(self, negative_vars=None, centered_vars=None, ref_values=None):
        self.negative_vars = negative_vars
        self.centered_vars = centered_vars
        self.ref_values = ref_values
    
    def fit(self, X, y=None):
        # Placeholder for potential diagnostic storage
        self.signs_ = {}
        return self
    
    def transform(self, X):
        X_copy = X.copy()

        # --- Invert variables with negative interpretation ---
        existing_negative_vars = [
            v for v in (self.negative_vars or []) 
            if v in X_copy.columns
        ]

        if existing_negative_vars:
            X_copy[existing_negative_vars] = -1 * X_copy[existing_negative_vars]

        # --- Center variables around reference values ---
        if self.centered_vars and self.ref_values:
            for var in self.centered_vars:
                if var in X_copy.columns:

                    # Store direction of deviation from reference
                    self.signs_[var] = np.sign(X_copy[var] - self.ref_values[var])

                    # Compute deviation from reference value
                    X_copy[var] = X_copy[var] - self.ref_values[var]

                    # Negate absolute deviation so that values closer to the reference score higher
                    X_copy[var] = -1 * np.abs(X_copy[var])

        return X_copy

# <span style="color:#e0bda8">5. Pre-Processing </span>

In [66]:
env_pipeline_mm = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')), 
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ("log1p", SignedLog1pTransformer(vars_to_transform=transformation_vars)),
    ('scaler', SafeMinMaxScaler(columns=df_env_filtered.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])

soc_pipeline_mm = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')), 
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ("log1p", SignedLog1pTransformer(vars_to_transform=transformation_vars)),
    ('scaler', SafeMinMaxScaler(columns=df_soc_filtered.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])

gov_pipeline_mm = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')), 
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ("log1p", SignedLog1pTransformer(vars_to_transform=transformation_vars)),
    ('scaler', SafeMinMaxScaler(columns=df_gov_filtered.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])

env_pipeline_yj = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')),
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ('yeo_johnson', SafePowerTransformer(columns=df_env_filtered.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])

soc_pipeline_yj = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')),
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ('yeo_johnson', SafePowerTransformer(columns=df_soc_filtered.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])

gov_pipeline_yj = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')), 
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ('yeo_johnson', SafePowerTransformer(columns=df_gov_filtered.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])



# Apply pipelines to the cleaned dataframes
env_mm = env_pipeline_mm.fit_transform(df_env_filtered)
df_env_mm = pd.DataFrame(
    env_mm,
    index=env_mm.index,
    columns=env_mm.columns
)

env_yj = env_pipeline_yj.fit_transform(df_env_filtered)
df_env_yj = pd.DataFrame(
    env_yj,
    index=env_yj.index,
    columns=env_yj.columns
)

soc_mm = soc_pipeline_mm.fit_transform(df_soc_filtered)
df_soc_mm = pd.DataFrame(
    soc_mm,
    index=soc_mm.index,
    columns=soc_mm.columns
)

soc_yj = soc_pipeline_yj.fit_transform(df_soc_filtered)
df_soc_yj = pd.DataFrame(
    soc_yj,
    index=soc_yj.index,
    columns=soc_yj.columns
)

gov_mm = gov_pipeline_mm.fit_transform(df_gov_filtered)
df_gov_mm = pd.DataFrame(
    gov_mm,
    index=gov_mm.index,
    columns=gov_mm.columns
)


gov_yj = gov_pipeline_yj.fit_transform(df_gov_filtered)
df_gov_yj = pd.DataFrame(
    gov_yj,
    index=gov_yj.index,
    columns=gov_yj.columns
)

In [67]:
df_env_mm.to_csv(os.path.join(DATA_PROCESSED, 'df_env_mm.csv'))
df_soc_mm.to_csv(os.path.join(DATA_PROCESSED, 'df_soc_mm.csv'))
df_gov_mm.to_csv(os.path.join(DATA_PROCESSED, 'df_gov_mm.csv'))

df_env_yj.to_csv(os.path.join(DATA_PROCESSED, 'df_env_yj.csv'))
df_soc_yj.to_csv(os.path.join(DATA_PROCESSED, 'df_soc_yj.csv'))
df_gov_yj.to_csv(os.path.join(DATA_PROCESSED, 'df_gov_yj.csv'))


# <span style="color:#e0bda8">6. Restoring the Dataset to its Original Scale </span>

### <span style="color:#e0bda8">6.0. Auxiliary Functions </span>

In [68]:
def signed_log1p_inverse(X, vars_to_transform):
    """
    Applies the inverse of the signed log1p transformation.

    The transformation being inverted is:
        signed_log1p(x) = sign(x) * log1p(|x|)

    This function restores the original scale:
        x = sign(y) * (exp(|y|) - 1)

    Parameters:
        X (DataFrame): Input dataset containing transformed variables
        vars_to_transform (list): List of variables to invert

    Returns:
        DataFrame: Dataset with selected variables transformed back
                  to their original scale
    """

    X_copy = X.copy()

    for var in vars_to_transform or []:
        if var not in X_copy.columns:
            continue

        # Apply inverse signed log1p transformation
        X_copy[var] = np.sign(X_copy[var]) * (np.expm1(np.abs(X_copy[var])))

    return X_copy


In [69]:
def signal_aligning_inverse(X, negative_vars=None, centered_vars=None, ref_values=None, signs=None):
    """
    Reverses the SignalAligningTransformer.

    - Restores original direction for negatively oriented variables
    - Reconstructs centered variables using stored reference values and signs

    Parameters:
        X (DataFrame): Transformed dataset
        negative_vars (list): Variables that were inverted
        centered_vars (list): Variables that were centered around a reference
        ref_values (dict): Reference values used during transformation
        signs (dict): Stored sign of deviation from reference

    Returns:
        DataFrame: Dataset restored to original signal orientation
    """

    X_copy = X.copy()
    
    # --- Restore original direction for negatively oriented variables ---
    existing_negative_vars = [
        v for v in (negative_vars or [])
        if v in X_copy.columns
    ]

    if existing_negative_vars:
        X_copy[existing_negative_vars] = -1 * X_copy[existing_negative_vars]
    
    # --- Reconstruct centered variables ---
    if centered_vars and ref_values:
        for var in centered_vars:
            if var in X_copy.columns:
                
                # Retrieve stored sign (default = 1 if not available)
                sign = signs.get(var, 1) if signs else 1

                # Reverse transformation:
                # original = reference + signed deviation
                X_copy[var] = ref_values[var] + (sign * (-1 * X_copy[var]))
               
    return X_copy

In [70]:
def inverse_pipeline(df_transformed, pipeline, vars_to_transform=None, negative_vars=None, centered_vars=None, ref_values=None):
    """
    Reconstructs original feature values by sequentially inverting
    transformations applied in the preprocessing pipeline.

    The inversion follows the reverse order of transformations:
    1. Scaling (MinMax / Standard / etc.)
    2. Signed Log1p transformation
    3. Signal alignment (direction + centering)

    Parameters:
        df_transformed (DataFrame): Transformed dataset
        pipeline (Pipeline): Fitted preprocessing pipeline
        vars_to_transform (list): Variables transformed via signed log1p
        negative_vars (list): Variables inverted during signal alignment
        centered_vars (list): Variables centered around reference values
        ref_values (dict): Reference values used for centering

    Returns:
        DataFrame: Approximation of the original dataset
    """

    X_inv = df_transformed.copy()
    
    # --- 1. Invert scaling transformation ---
    if 'scaler' in pipeline.named_steps:
        scaler_step = pipeline.named_steps['scaler']

        # Apply inverse scaling only to scaled columns
        X_inv[scaler_step.columns] = scaler_step.scaler.inverse_transform(
            X_inv[scaler_step.columns]
        )
    
    # --- 2. Invert signed log1p transformation ---
    X_inv = signed_log1p_inverse(X_inv, vars_to_transform)

    # --- 3. Invert signal alignment transformation ---
    signs = {}
    if 'signal_aligning' in pipeline.named_steps:
        # Retrieve stored sign information from fitted transformer
        signs = pipeline.named_steps['signal_aligning'].signs_

    X_inv = signal_aligning_inverse(
        X_inv,
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values,
        signs=signs
    )

    return X_inv

## <span style="color:#e0bda8">6.1.  Final Datasets on the Original Scale </span>

In [71]:
df_env_unscaled = inverse_pipeline(
    df_env_mm, 
    env_pipeline_mm, 
    vars_to_transform=transformation_vars, 
    negative_vars=negative_vars,
    centered_vars=centered_vars, 
    ref_values=ref_values
)

df_soc_unscaled = inverse_pipeline(
    df_soc_mm, 
    soc_pipeline_mm, 
    vars_to_transform=transformation_vars, 
    negative_vars=negative_vars,
    centered_vars=centered_vars, 
    ref_values=ref_values
)

df_gov_unscaled = inverse_pipeline(
    df_gov_mm, 
    gov_pipeline_mm, 
    vars_to_transform=transformation_vars, 
    negative_vars=negative_vars,
    centered_vars=centered_vars, 
    ref_values=ref_values
)

# <span style="color:#e0bda8">7. Summary Statistics </span>

In [72]:
def full_summary(df):
    """
    Generates a comprehensive summary of dataset variables.

    For numeric variables:
    - Computes central tendency, dispersion, distribution shape, and range

    For non-numeric variables:
    - Reports only variable name and data type

    Returns:
        DataFrame with descriptive statistics per variable
    """

    summary_list = []
    
    for col in df.columns:
        dtype = df[col].dtype
        
        # --- Numeric variables: compute full statistical profile ---
        if pd.api.types.is_numeric_dtype(df[col]):
            col_mean = df[col].mean()
            col_std = df[col].std()
            col_min = df[col].min()
            col_25 = df[col].quantile(0.25)
            col_50 = df[col].quantile(0.5)
            col_75 = df[col].quantile(0.75)
            col_max = df[col].max()

            # Distribution shape metrics
            col_skew = skew(df[col].dropna())
            col_kurt = kurtosis(df[col].dropna())

        # --- Non-numeric variables: statistics not applicable ---
        else:
            col_mean = col_std = col_min = col_25 = col_50 = col_75 = col_max = col_skew = col_kurt = None
        
        summary_list.append({
            'Variable': col,
            'Data type': dtype,
            'Mean': col_mean,
            'Std': col_std,
            'Min': col_min,
            '25%': col_25,
            '50%': col_50,
            '75%': col_75,
            'Max': col_max,
            'Skewness': col_skew,
            'Kurtosis': col_kurt
        })
    
    summary_df = pd.DataFrame(summary_list)

    # Round numeric values for cleaner presentation
    return summary_df.round(2)

In [73]:
summary_env = full_summary(df_env_unscaled)
summary_soc = full_summary(df_soc_unscaled)
summary_gov = full_summary(df_gov_unscaled)

summary =  pd.concat(
    [summary_env, summary_soc, summary_gov],
    keys=['Environment', 'Social', 'Governance']
)

In [74]:
pd.set_option('display.max_rows', None)

summary

Variable Data type  \
Environment 0   PM2.5 air pollution, mean annual exposure (mic...   float64   
            1   Carbon dioxide (CO2) net fluxes from LULUCF - ...   float64   
            2   Total greenhouse gas emissions per capita excl...   float64   
            3   Electricity production from coal sources (% of...   float64   
            4               Energy imports, net (% of energy use)   float64   
            5   Energy intensity level of primary energy (MJ/$...   float64   
            6        Energy use (kg of oil equivalent per capita)   float64   
            7         Fossil fuel energy consumption (% of total)   float64   
            8   Renewable electricity output (% of total elect...   float64   
            9   Renewable energy consumption (% of total final...   float64   
            10                                Cooling Degree Days   float64   
            11                                Heating Degree Days   float64   
            12  Level of water stress: freshwater withdrawal a...   float64   
            13  Standardised Precipitation-Evapotranspiration ...   float64   
            14  Agriculture, forestry, and fishing, value adde...   float64   
            15            Food production index (2014-2016 = 100)   float64   
            16  Adjusted savings, natural resources depletion ...   float64   
            17  Adjusted savings, net forest depletion (% of GNI)   float64   
            18  Annual freshwater withdrawals, total (% of int...   float64   
            19                       Forest area (% of land area)   float64   
            20  Terrestrial and marine protected areas (% of t...   float64   
            21                                    Tree Cover Loss   float64   
Social      0   Access to clean fuels and technologies for coo...   float64   
            1   People using safely managed drinking water ser...   float64   
            2   People using safely managed sanitation service...   float64   
            3            Fertility rate, total (births per woman)   float64   
            4             Life expectancy at birth, total (years)   float64   
            5   Government expenditure on education, total (% ...   float64   
            6                School enrollment, primary (% gross)   float64   
            7   Labor force participation rate, total (% of to...   float64   
            8   Share of youth not in education, employment or...   float64   
            9   Unemployment, total (% of total labor force) (...   float64   
            10  Wage and salaried workers, total (% of total e...   float64   
            11                   Hospital beds (per 1,000 people)   float64   
            12    Mortality rate, under-5 (per 1,000 live births)   float64   
            13             Prevalence of overweight (% of adults)   float64   
            14   Prevalence of undernourishment (% of population)   float64   
            15                                         Gini index   float64   
            16  Poverty headcount ratio at $3.00 a day (2021 P...   float64   
            17  Poverty headcount ratio at $8.30 a day (2021 P...   float64   
            18  Poverty headcount ratio at national poverty li...   float64   
Governance  0    Individuals using the Internet (% of population)   float64   
            1   Proportion of seats held by women in national ...   float64   
            2   Ratio of female to male labor force participat...   float64   
            3                  Government Effectiveness: Estimate   float64   
            4        Economic and Social Rights Performance Score   float64   
            5                  Voice and Accountability: Estimate   float64   
            6                      Patent applications, residents   float64   
            7     Research and development expenditure (% of GDP)   float64   
            8                                       Net migration   float64   
            9   Political St

In [75]:
summary.to_excel(os.path.join(DATA_STATS, "summary_statistics.xlsx"))